In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_moons
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeRegressor

print('Good imports')


In [ ]:
iris = load_iris()

X = iris.data[:, [2, 3]]
y = iris.target

tree_clf = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_clf.fit(X, y)

plt.figure(figsize=(10,8))
plot_tree(tree_clf,
        feature_names=['Длина лепестка (см)', 'Ширина лепестка (см)'],
        class_names=iris.target_names,
        filled = True,
        rounded=True)
plt.title('Визуализация дерева решений на датасете Iris')

In [ ]:
def plot_decision_boundary(clf, X, y):
    x1s = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
    x2s = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 100)
    x1, x2 = np.meshgrid(x1s, x2s)
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)
    
    plt.contourf(x1, x2, y_pred, alpha=0.3, cmap='brg')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='brg', edgecolor='k', s=40)
    plt.xlabel("Длина лепестка (см)")
    plt.ylabel("Ширина лепестка (см)")

plt.figure(figsize=(9, 6))
plot_decision_boundary(tree_clf, X, y)

plt.axvline(x=2.45, color='k', linestyle='-', linewidth=2)
plt.axhline(y=1.75, color='k', linestyle='--', linewidth=2)

plt.title("Границы решения")
plt.show()

In [ ]:
# Создаем искусственный цветок
test_flower = [[5, 1.5]]
probabilities = tree_clf.predict_proba(test_flower) # точные вероятности для каждого вида
predicted_class = tree_clf.predict(test_flower)

print("--- Оценка вероятностей классов ---")
for i, name in enumerate(iris.target_names):
    print(f"Вероятность сорта {name.capitalize()}: {probabilities[0][i]:.2%}")

print(f"\nФинальный Класс: {iris.target_names[predicted_class][0].capitalize()}")


In [ ]:
feature_importances = tree_clf.feature_importances_

print("--- Важность признаков по алгоритму CART ---")
for name, importance in zip(["Длина лепестка", "Ширина лепестка"], feature_importances):
    print(f"Признак '{name}': важность = {importance:.2%}")


In [ ]:
large_test_data = np.random.rand(10000, 2) * 5

print("Измеряем скорость предсказания для 10 000 объектов:")
%timeit tree_clf.predict(large_test_data)

In [ ]:
# Обучаем дерево с критерием Энтропии
entropy_tree_clf = DecisionTreeClassifier(max_depth=2, criterion="entropy", random_state=42)
entropy_tree_clf.fit(X, y)

plt.figure(figsize=(10, 8))
plot_tree(entropy_tree_clf, 
          feature_names=["Длина лепестка (см)", "Ширина лепестка (см)"], 
          class_names=iris.target_names, 
          filled=True, 
          rounded=True)
plt.title("Дерево решений с критерием Энтропии (Entropy)")
plt.show()

In [ ]:
X_moons, y_moons = make_moons(n_samples=150, noise=0.3, random_state=42)

tree_clf_unregulated = DecisionTreeClassifier(random_state=42)
tree_clf_unregulated.fit(X_moons, y_moons)

tree_clf_regulated = DecisionTreeClassifier(min_samples_leaf=5, random_state=42)
tree_clf_regulated.fit(X_moons, y_moons)

def plot_predictions(clf, axes):
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_new = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X_new).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, alpha=0.3, cmap="Wistia")
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="brg", edgecolor='k', s=30)
    plt.axis(axes)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plot_predictions(tree_clf_unregulated, [-1.5, 2.5, -1, 1.5])
plt.title("Без регуляризации")

plt.subplot(1, 2, 2)
plot_predictions(tree_clf_regulated, [-1.5, 2.5, -1, 1.5])
plt.title("С регуляризацией")

plt.show()


In [ ]:
np.random.seed(42)
X_reg = np.random.rand(80, 1) * 5 - 2.5
y_reg = X_reg**2 + np.random.randn(80, 1) * 0.5

tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg2 = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_reg1.fit(X_reg, y_reg)
tree_reg2.fit(X_reg, y_reg)

X_grid = np.linspace(-2.5, 2.5, 500).reshape(-1, 1)
y_pred1 = tree_reg1.predict(X_grid)
y_pred2 = tree_reg2.predict(X_grid)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_reg, y_reg, color="blue", s=30)
plt.plot(X_grid, y_pred1, "r-", linewidth=3, label="max_depth=2")
plt.xlabel("Признак X")
plt.ylabel("Целевая переменная y")
plt.title("Регрессия 1")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(X_reg, y_reg, color="blue", s=30)
plt.plot(X_grid, y_pred2, "g-", linewidth=3, label="max_depth=4")
plt.xlabel("Признак X")
plt.title("Регрессия 2")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
widest_iris_index = np.argmax(X[:, 1])

X_tweaked = np.delete(X, widest_iris_index, axis=0)
y_tweaked = np.delete(y, widest_iris_index, axis=0)

tree_clf_tweaked = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_clf_tweaked.fit(X_tweaked, y_tweaked)

plt.figure(figsize=(9, 6))
plot_decision_boundary(tree_clf_tweaked, X_tweaked, y_tweaked)
plt.title("Границы решения после УДАЛЕНИЯ ВСЕГО ОДНОЙ точки!")
plt.show()
